# 11 · Robustness — which conclusions survive?
Each check is re-computed; criteria are explicit in the classification table.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4g}")
from src import pipeline as P
from src.config import *

In [2]:
ctx = P.prepare('stage_robustness')
ctx = P.stage_robustness(ctx)
r = ctx['robust']
r['rfm_bins']

[19:45:18] robustness


,n_bins,segment_ARI_vs_quintiles,high_value_jaccard_vs_quintiles,rfm_score_auc_on_pilot,pilot_response_top20pct_by_rfm
0,3,0.7603,0.8591,0.7034,0.3102
1,4,0.7049,0.8338,0.7214,0.3102
2,5,1,1,0.7224,0.3201
3,10,0.8279,0.9423,0.7253,0.3176


In [3]:
r['kmeans_k']

,k,ARI_vs_selected,response_cramers_v,response_rate_range
0,3,0.5085,0.1933,0.1552
1,4,0.4644,0.3578,0.546
2,5,0.3465,0.3583,0.5643
3,6,0.3355,0.3691,0.61


In [4]:
r['train_window']

,training_campaigns,train_rows,roc_auc,pr_auc,lift@10%,recall@10%,lift@20%,recall@20%,lift@30%,recall@30%
0,C5 only,2019,0.7045,0.4112,3.434,0.3435,2.465,0.4932,1.949,0.585
1,C4-C5,4038,0.6661,0.3486,2.992,0.2993,2.261,0.4524,1.813,0.5442
2,C2-C5 (primary),8076,0.7401,0.412,3.23,0.3231,2.329,0.466,2.096,0.6293
3,C2-C5 excl. C3,6057,0.6761,0.3594,3.026,0.3027,2.261,0.4524,1.847,0.5544


In [5]:
r['drift']

,held_out_campaign,acceptance_rate,roc_auc,pr_auc,rule_auc_prior_acceptances
0,C2,0.01288,0.8794,0.09093,0.7004
1,C3,0.07182,0.6512,0.1161,0.5504
2,C4,0.07528,0.6524,0.1466,0.6115
3,C5,0.07231,0.8472,0.3786,0.7749
4,C6,0.1456,0.7401,0.412,0.7276


In [6]:
r['dedup']

,table,customers,pilot_response_rate,pilot_profit_mu,rule_auc_prior_acceptances,response_rate_if_prior_acceptor,response_rate_if_no_prior
0,"raw (2,240 incl. duplicates)",2240,0.1491,-3046,0.7225,0.406,0.08216
1,clean (de-duplicated),2019,0.1456,-2823,0.7276,0.4029,0.07865


In [7]:
r['uplift_boot']

,conclusion,share_of_bootstrap_resamples_true
0,mens_gt_womens_spend,0.9833
1,womens_gt_zero_spend,0.9967
2,womens_on_mens_only_conv_gt0,0.71
3,mens_on_mens_only_conv_gt0,1


In [8]:
r['classification']

,finding,test,stable,classification
0,Prior campaign acceptance is the strongest single predictor of pilot response,top permutation importance; rule AUC under every de-dup/training window variant,True,STABLE
1,Model beats the prior-acceptance rule,"paired bootstrap PR-AUC diff CI [-0.013, 0.043] must exclude 0",False,SENSITIVE / NOT CONFIRMED
2,Model ranking is better than random on a later campaign,ROC-AUC CI lower bound > 0.5 for all training windows,True,STABLE
3,Targeting the top 10-20% by model is profitable in the back-test,bootstrap P(profit>0) at 10% and 20% contact share > 0.95,True,STABLE
4,Mass contact loses money,"true for 8/9 cost/revenue combinations (cost 2-4, revenue 8-14 MU)",False,SENSITIVE / NOT CONFIRMED
5,Profit-optimal contact share,hindsight-optimal share ranges 5.0%-29.5% across cost/revenue grid,False,SENSITIVE / NOT CONFIRMED
6,RFM value tier membership,Jaccard of 'High value' tier vs quintile scoring min=0.83,True,STABLE
7,Exact 3x3 RFM segment membership,ARI vs quintile scoring min=0.70,False,SENSITIVE / NOT CONFIRMED
8,KMeans segment structure,"ARI of alternative k vs selected: k=3:0.51, k=4:0.46, k=5:0.35, k=6:0.34",False,SENSITIVE / NOT CONFIRMED
9,Pilot recency effect is real behaviour,Recency AUC is ~0.5 on C1-C5 but 0.66 on C6 -> timing artefact suspected,False,SENSITIVE / NOT CONFIRMED


### Temporal stability / drift
Campaign acceptance varies from 1.3 % (C2) to 14.6 % (C6) and the model's held-out ROC-AUC from 0.65 (C3, C4) to 0.88,
so **performance drifts with campaign content**. There are no dates, so behaviour change over calendar time cannot be measured;
enrolment cohorts are the only time axis (notebook 08).

### Summary
**Stable:** campaign history is the strongest predictor; model ranking beats random on a later campaign; targeting a small top slice
is profitable in the back-test; the high-value tier membership; leakage-prone features do not drive the temporal model; Mens > Womens e-mail.
**Sensitive / not confirmed:** model > rule; the exact optimal contact share; exact 3×3 RFM membership; KMeans structure;
the pilot recency effect; subgroup-specific e-mail effects; mass contact always losing (fails when revenue is 14 MU and cost 2 MU).